# SMC/ICT Strategy Backtest Analysis

This notebook demonstrates how to run the SMC/ICT reversal strategy
using the custom backtest engine and generate visualization reports.

**Note:** The SMC strategy requires 5-minute or finer intraday data.

---

## Quick Configuration Guide

Modify the `CONFIG` dictionary in Section 1 to customize:
- Data source and date range
- SMC strategy parameters (session times, ATR settings)
- Risk management parameters
- Output settings

---

## 1. Configuration Section

**Modify parameters below to customize the SMC backtest.**

In [1]:
# ============================================================
# CONFIGURATION - Modify these parameters to customize analysis
# ============================================================

CONFIG = {
    # ----------------------------------------------------------
    # Data Configuration
    # ----------------------------------------------------------
    "data": {
        "file": "SPY_5min.csv",  # Primary: 5-minute data
        "fallback_file": "SPY_daily.csv",  # Fallback: daily data
        "directory": "data/raw",
        "start_date": None,
        "end_date": None,
        "columns": ["Open", "High", "Low", "Close", "Volume"],
    },
    # ----------------------------------------------------------
    # SMC Strategy Configuration
    # ----------------------------------------------------------
    "smc": {
        # Session times (UTC)
        "session_start": "00:00",  # Asian session start
        "session_end": "08:00",  # Asian session end
        # ATR settings
        "atr_period": 14,
        "atr_buffer_mult": 0.5,
        "ifvg_atr_mult": 1.2,
        "ifvg_proximity_mult": 1.5,
        # Risk management
        "risk_per_trade": 0.01,  # 1% risk per trade
        "slippage_buffer": 0.1,
        # Targets
        "target_1r": 1.0,  # First target at 1R (breakeven)
        "target_2r": 2.0,  # Second target at 2R
        "target_final": 2.5,  # Final target at 2.5R
        # Daily limits
        "daily_loss_limit": 0.03,  # 3% daily loss limit
        "max_trades_per_day": 3,
        # Confirmations
        "require_volume_confirmation": True,
        "require_mss_confirmation": True,
    },
    # ----------------------------------------------------------
    # Backtest Configuration
    # ----------------------------------------------------------
    "backtest": {
        "initial_equity": 100000,
        "commission_pct": 0.001,  # 0.1% commission
        "slippage_pct": 0.0005,  # 0.05% slippage
        "risk_per_trade": 0.01,
        "max_open_positions": 10,  # SMC typically trades one position
        "min_confidence": 0.5,
    },
    # ----------------------------------------------------------
    # Output Configuration
    # ----------------------------------------------------------
    "output": {
        "directory": "reports",
        "save_plots": True,
        "show_plots": True,
        "dpi": 150,
    },
}

---

## 2. Setup and Imports

In [2]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Setup project root FIRST (before any src imports)
candidates = [
    Path(".").resolve(),
    Path(".").resolve(),
]
project_root = None
for root in candidates:
    if (root / "src").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        project_root = root
        break
if project_root is None:
    project_root = Path(".").resolve()
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

# Import notebook helpers
from src.utils.notebook_helpers import (
    load_price_data,
    print_data_summary,
)

# Standard imports

# Import SMC strategy
from src.strategies import SMCReversalStrategy, SMCConfig
from src.indicators.asian_range import detect_asian_range
from src.indicators.ifvg import detect_ifvg
from src.indicators.mss import detect_mss
from src.indicators.technical import atr as atr_indicator

print("✅ Imports successful!")
print(f"Project root: {project_root}")

Loading BokehJS ...

✅ Imports successful!
Project root: C:\Dev\projects\investment_trying


---

## 3. Load and Prepare Data

The SMC strategy requires intraday data (5-minute bars recommended).

In [3]:
from pathlib import Path

# Force loading daily data since 5min isn't available
data_path = project_root / CONFIG["data"]["directory"] / "SPY_daily.csv"
assert data_path.exists(), "Need data file for SMC notebook"

print("WARNING: 5-minute data not found. Using daily data for demonstration.")
print("For proper SMC backtesting, please provide 5-minute OHLCV data.")

fallback_config = CONFIG.copy()
fallback_config["data"] = CONFIG["data"].copy()
fallback_config["data"]["file"] = "SPY_daily.csv"
df = load_price_data(fallback_config, project_root)
data_freq = "daily"

# Display data summary
print_data_summary(df, title=f"SMC Backtest Data ({data_freq})")

# Check data frequency
median_diff = df.index.to_series().diff().median()
print(f"Data frequency: {median_diff}")

For proper SMC backtesting, please provide 5-minute OHLCV data.
📊 SMC Backtest Data (daily)
Date Range: 2015-01-02 to 2024-12-30
Total Trading Days: 2,515
Years of Data: 10.0

Columns: ['Open', 'High', 'Low', 'Close', 'Volume']

Data shape: (2515, 5)

First 5 rows:
                  Open        High         Low       Close     Volume
Date                                                                 
2015-01-02  170.911759  171.325830  169.089839  170.125015  121465900
2015-01-05  169.081555  169.247181  166.746204  167.052612  169632600
2015-01-06  167.359012  167.880745  164.684120  165.479141  209151400
2015-01-07  166.804154  167.880740  166.356963  167.541199  125346700
2015-01-08  168.949050  170.729561  168.932497  170.514236  147217800

📈 Price Statistics:
          Open     High      Low    Close
count  2515.00  2515.00  2515.00  2515.00
mean    309.32   310.99   307.49   309.36
std     114.06   114.62   113.43   114.07
min     154.12   155.61   152.47   154.56
25%     211.6

---

## 4. SMC Strategy Components

Let's examine the SMC indicators individually.

In [4]:
# Detect Asian Range (for 5-minute data)
# The Asian session is typically 00:00-08:00 UTC

if data_freq == "5-minute":
    print("Detecting Asian Range...")
    asian_range = detect_asian_range(df)
    if asian_range:
        print(f"Asian Range High: {asian_range.high}")
        print(f"Asian Range Low: {asian_range.low}")
        print(f"Range Size: {asian_range.range_size}")
        print(f"Is Low Volatility: {asian_range.is_low_vol}")
else:
    print("Daily data detected - Asian Range detection requires intraday data")

Daily data detected - Asian Range detection requires intraday data


In [5]:
# Detect IFVG (Inverse Fair Value Gaps)
print("Detecting IFVGs...")
atr_series = atr_indicator(df, period=14)
ifvg_list = detect_ifvg(df, atr=atr_series)

print(f"Found {len(ifvg_list)} IFVGs")
if ifvg_list:
    print("\nRecent IFVGs:")
    for ifvg in ifvg_list[:5]:
        print(f"  Direction: {ifvg.direction}, Range: [{ifvg.low:.2f}, {ifvg.high:.2f}]")
        print(f"    Filled: {ifvg.filled}, Gap Size: {ifvg.gap_size:.2f}")

Detecting IFVGs...


2026-05-01 15:29:49.794 | DEBUG    | src.indicators.ifvg:detect_ifvg:188 - Bullish IFVG detected at bar 108: zone=173.9372-175.6841, size=1.7470


2026-05-01 15:29:49.811 | DEBUG    | src.indicators.ifvg:detect_ifvg:209 - Bearish IFVG detected at bar 158: zone=170.4702-173.3206, size=2.8504


2026-05-01 15:29:49.813 | DEBUG    | src.indicators.ifvg:detect_ifvg:209 - Bearish IFVG detected at bar 159: zone=165.0704-170.4368, size=5.3664


2026-05-01 15:29:49.831 | DEBUG    | src.indicators.ifvg:detect_ifvg:209 - Bearish IFVG detected at bar 217: zone=171.9697-174.4821, size=2.5123


...

2026-05-01 15:29:50.562 | DEBUG    | src.indicators.ifvg:detect_ifvg:209 - Bearish IFVG detected at bar 2410: zone=512.7894-528.3127, size=15.5233


2026-05-01 15:29:50.577 | DEBUG    | src.indicators.ifvg:detect_ifvg:188 - Bullish IFVG detected at bar 2476: zone=562.4209-575.0842, size=12.6633


2026-05-01 15:29:50.579 | DEBUG    | src.indicators.ifvg:detect_ifvg:188 - Bullish IFVG detected at bar 2477: zone=566.5864-582.5602, size=15.9738


2026-05-01 15:29:50.592 | DEBUG    | src.indicators.ifvg:detect_ifvg:209 - Bearish IFVG detected at bar 2506: zone=582.5600-592.2760, size=9.7159


Found 61 IFVGs

Recent IFVGs:
  Direction: bullish, Range: [173.94, 175.68]
    Filled: False, Gap Size: 1.75
  Direction: bearish, Range: [170.47, 173.32]
    Filled: False, Gap Size: 2.85
  Direction: bearish, Range: [165.07, 170.44]
    Filled: False, Gap Size: 5.37
  Direction: bearish, Range: [171.97, 174.48]
    Filled: False, Gap Size: 2.51
  Direction: bearish, Range: [169.92, 173.91]
    Filled: False, Gap Size: 4.00


In [6]:
# Detect Market Structure Shifts (limited sample for daily data)
print("Detecting Market Structure Shifts (sampling every 10th day)")
mss_list = []

# For daily data, sample every 10th bar to keep it fast
step = 10 if data_freq == "daily" else 1
for i in range(50, len(df), step):
    mss = detect_mss(df, i)
    if mss.detected and getattr(mss, "is_valid", False):
        mss_list.append(
            {
                "index": i,
                "timestamp": df.index[i],
                "direction": mss.direction,
                "break_price": mss.break_price,
            }
        )

print(f"Found {len(mss_list)} valid MSS signals")
if mss_list:
    print("\nRecent MSS signals:")
    for mss in mss_list[-5:]:
        print(f"  {mss['timestamp']}: {mss['direction']} at {mss['break_price']:.2f}")

Detecting Market Structure Shifts (sampling every 10th day)


2026-05-01 15:29:51.030 | DEBUG    | src.indicators.mss:detect_mss:249 - Bullish MSS detected: close 444.9999 > pivot high 443.7642 at bar 2244


2026-05-01 15:29:51.405 | DEBUG    | src.indicators.mss:detect_mss:249 - Bullish MSS detected: close 444.9999 > pivot high 443.7642 at bar 2244


2026-05-01 15:29:51.757 | DEBUG    | src.indicators.mss:detect_mss:249 - Bullish MSS detected: close 444.9999 > pivot high 443.7642 at bar 2244


2026-05-01 15:29:52.137 | DEBUG    | src.indicators.mss:detect_mss:249 - Bullish MSS detected: close 444.9999 > pivot high 443.7642 at bar 2244


...

2026-05-01 15:30:21.180 | DEBUG    | src.indicators.mss:detect_mss:249 - Bullish MSS detected: close 456.0498 > pivot high 452.7858 at bar 2252


2026-05-01 15:30:21.507 | DEBUG    | src.indicators.mss:detect_mss:249 - Bullish MSS detected: close 456.0498 > pivot high 452.7858 at bar 2252


2026-05-01 15:30:21.798 | DEBUG    | src.indicators.mss:detect_mss:249 - Bullish MSS detected: close 456.0498 > pivot high 452.7858 at bar 2252


2026-05-01 15:30:22.570 | DEBUG    | src.indicators.mss:detect_mss:249 - Bullish MSS detected: close 456.0498 > pivot high 452.7858 at bar 2252


Found 44 valid MSS signals

Recent MSS signals:
  2016-09-30 00:00:00: bullish at 456.05
  2016-10-14 00:00:00: bullish at 456.05
  2016-10-28 00:00:00: bullish at 456.05
  2016-11-11 00:00:00: bullish at 456.05
  2016-11-28 00:00:00: bullish at 456.05


---

## 5. Configure SMC Strategy

In [7]:
# Create SMC configuration from CONFIG dict
smc_params = CONFIG["smc"]

smc_config = SMCConfig(
    session_start=smc_params["session_start"],
    session_end=smc_params["session_end"],
    atr_period=smc_params["atr_period"],
    atr_buffer_mult=smc_params["atr_buffer_mult"],
    ifvg_atr_mult=smc_params["ifvg_atr_mult"],
    ifvg_proximity_mult=smc_params["ifvg_proximity_mult"],
    risk_per_trade=smc_params["risk_per_trade"],
    slippage_buffer=smc_params["slippage_buffer"],
    target_1r=smc_params["target_1r"],
    target_2r=smc_params["target_2r"],
    target_final=smc_params["target_final"],
    daily_loss_limit=smc_params["daily_loss_limit"],
    max_trades_per_day=smc_params["max_trades_per_day"],
    require_volume_confirmation=smc_params["require_volume_confirmation"],
    require_mss_confirmation=smc_params["require_mss_confirmation"],
)

print("SMC Strategy Configuration:")
print(f"  Session: {smc_config.session_start} - {smc_config.session_end} UTC")
print(f"  Risk per trade: {smc_config.risk_per_trade * 100}%")
print(f"  Daily loss limit: {smc_config.daily_loss_limit * 100}%")
print(f"  Max trades per day: {smc_config.max_trades_per_day}")

SMC Strategy Configuration:
  Session: 00:00 - 08:00 UTC
  Risk per trade: 1.0%
  Daily loss limit: 3.0%
  Max trades per day: 3


---

## 6. Run Backtest with Custom Engine

In [8]:
# Configure backtest
backtest_params = CONFIG["backtest"]

print("Backtest Configuration:")
print(f"  Initial equity: ${backtest_params['initial_equity']:,.0f}")
print(f"  Commission: {backtest_params['commission_pct'] * 100:.2f}%")
print(f"  Slippage: {backtest_params['slippage_pct'] * 100:.3f}%")

Backtest Configuration:
  Initial equity: $100,000
  Commission: 0.10%
  Slippage: 0.050%


In [9]:
# Initialize strategy
smc_strategy = SMCReversalStrategy(config=smc_config)

print("✅ Strategy initialized")

2026-05-01 15:39:22.331 | INFO     | src.strategies.smc_reversal:__init__:237 - SMCReversalStrategy initialized with config: session=00:00-08:00, risk=1.0%


✅ Strategy initialized


In [10]:
# Run backtest
print("Running SMC backtest...")

# Note: SMC strategy works best on 5-minute data
# Results on daily data will be limited

signals = smc_strategy.run(df)

print("\nBacktest complete!")
print(f"Total signals: {len(signals)}")

2026-05-01 15:39:22.364 | INFO     | src.strategies.smc_reversal:run:276 - Running SMC strategy on 2515 trading days


2026-05-01 15:39:22.369 | DEBUG    | src.strategies.smc_reversal:_handle_session_start:345 - Session start at 2015-01-02 00:00:00+00:00


2026-05-01 15:39:22.372 | DEBUG    | src.strategies.smc_reversal:_handle_session_start:345 - Session start at 2015-01-05 00:00:00+00:00


2026-05-01 15:39:22.374 | DEBUG    | src.strategies.smc_reversal:_handle_session_start:345 - Session start at 2015-01-06 00:00:00+00:00


2026-05-01 15:39:22.376 | DEBUG    | src.strategies.smc_reversal:_handle_session_start:345 - Session start at 2015-01-06 00:00:00+00:00


...

2026-05-01 15:39:32.139 | DEBUG    | src.strategies.smc_reversal:_handle_session_start:345 - Session start at 2024-12-27 00:00:00+00:00


2026-05-01 15:39:32.141 | DEBUG    | src.strategies.smc_reversal:_handle_session_start:345 - Session start at 2024-12-27 00:00:00+00:00


2026-05-01 15:39:32.142 | DEBUG    | src.strategies.smc_reversal:_handle_session_start:345 - Session start at 2024-12-30 00:00:00+00:00


2026-05-01 15:39:32.143 | INFO     | src.strategies.smc_reversal:run:286 - Strategy complete: 0 signals generated



Backtest complete!
Total signals: 0


---

## 7. Analyze Results

In [11]:
# Display metrics
if signals:
    print("=" * 60)
    print("SMC STRATEGY PERFORMANCE")
    print("=" * 60)

    print(f"Total signals: {len(signals)}")
    long_signals = sum(1 for s in signals if s.direction == "long")
    short_signals = sum(1 for s in signals if s.direction == "short")
    print(f"Long signals: {long_signals}")
    print(f"Short signals: {short_signals}")
    print("=" * 60)

In [12]:
# Display signals
if signals:
    print("\n📊 Recent Signals:")
    for signal in signals[:10]:
        print(f"  {signal.timestamp}: {signal.direction} @ {signal.entry_price:.4f}")
else:
    print("\n⚠️ No signals generated. This is expected with daily data.")
    print("   SMC strategy requires 5-minute or finer intraday data.")


⚠️ No signals generated. This is expected with daily data.
   SMC strategy requires 5-minute or finer intraday data.


In [13]:
# Display signal details
if signals:
    print("\n📊 Signal Details:")
    for signal in signals[:5]:
        print(f"  {signal.timestamp}: {signal.direction}")
        print(f"    Entry: {signal.entry_price:.4f}, Stop: {signal.stop_loss:.4f}")
        print(
            f"    Targets: {signal.target_1:.4f}, {signal.target_2:.4f}, {signal.target_final:.4f}"
        )

---

## 8. Summary

In [14]:
print("\n" + "=" * 60)
print("📊 SMC BACKTEST SUMMARY")
print("=" * 60)
print(f"\nData frequency: {data_freq}")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")
print(f"Total bars: {len(df):,}")

if data_freq == "daily":
    print("\n⚠️ NOTE: SMC strategy is designed for intraday data.")
    print("   For meaningful results, please provide 5-minute OHLCV data.")
    print("   The strategy looks for:")
    print("   - Asian Range liquidity sweeps")
    print("   - Inverse Fair Value Gaps (IFVG)")
    print("   - Market Structure Shifts (MSS)")
else:
    print(f"\nTotal signals: {len(signals)}")

    if signals:
        print("\nKey Metrics:")
        long_signals = sum(1 for s in signals if s.direction == "long")
        short_signals = sum(1 for s in signals if s.direction == "short")
        print(f"  Long signals: {long_signals}")
        print(f"  Short signals: {short_signals}")
        avg_confidence = sum(s.confidence for s in signals) / len(signals)
        print(f"  Avg confidence: {avg_confidence:.2f}")

print("\n" + "=" * 60)


📊 SMC BACKTEST SUMMARY

Data frequency: daily
Date range: 2015-01-02 to 2024-12-30
Total bars: 2,515

⚠️ NOTE: SMC strategy is designed for intraday data.
   For meaningful results, please provide 5-minute OHLCV data.
   The strategy looks for:
   - Asian Range liquidity sweeps
   - Inverse Fair Value Gaps (IFVG)
   - Market Structure Shifts (MSS)

